# Known Projects → Residential Zoned Units Summary

Transforms `known_projects.csv` into the `Jurisdiction / Unit_Pool / Future_Units` schema
used by `forecast_residential_zoned_units.csv`.

**Input:** `Alternative_1/inputs/known_projects.csv`  
**Output:** `Alternative_1/inputs/known_projects_residential_summary.csv`

In [1]:
import pandas as pd
from pathlib import Path

# Paths
BASE = Path("Lookup_Lists")
INPUT_FILE  = BASE / "known_projects.csv"
OUTPUT_FILE = BASE / "known_projects_residential_summary.csv"

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df):,} rows")
df.head()

Loaded 310 rows


,APN,Quantity,Development Right,Development Type
0,014-242-008,1,Residential Allocation - El Dorado County,New Single-Family Residential ADU from Allocation
1,014-282-014,1,Residential Allocation - El Dorado County,New Single-Family Residential from Allocation
2,015-032-013,1,Residential Allocation - El Dorado County,New Single-Family Residential From Allocation
3,015-104-021,1,Residential Allocation,New Single-Family Residential From Allocation
4,015-152-006,1,Residential Allocation - El Dorado County,New Single-Family Residential from Allocation


## Step 1 – Filter: keep only residential development rights with positive quantities

In [2]:
RESIDENTIAL_RIGHTS = [
    "Residential Allocation",
    "Residential Bonus Unit (RBU)",
    "Single-Family Residential Unit of Use (SFRUU)",
    "Multi-Family Residential Unit of Use (MFRUU)",
    "Potential Residential Unit of Use (PRUU)",
]

mask_residential = df["Development Right"].str.strip().apply(
    lambda r: any(r.startswith(k) for k in RESIDENTIAL_RIGHTS)
)

# Remove comma-formatted numbers then cast to numeric
df["Quantity"] = pd.to_numeric(
    df["Quantity"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

res = df[mask_residential & (df["Quantity"] > 0)].copy()
print(f"Residential rows with positive quantity: {len(res):,}")
res.head()

Residential rows with positive quantity: 261


,APN,Quantity,Development Right,Development Type
0,014-242-008,1,Residential Allocation - El Dorado County,New Single-Family Residential ADU from Allocation
1,014-282-014,1,Residential Allocation - El Dorado County,New Single-Family Residential from Allocation
2,015-032-013,1,Residential Allocation - El Dorado County,New Single-Family Residential From Allocation
3,015-104-021,1,Residential Allocation,New Single-Family Residential From Allocation
4,015-152-006,1,Residential Allocation - El Dorado County,New Single-Family Residential from Allocation


## Step 2 – Map Jurisdiction

Priority:
1. Explicit county/city name in `Development Right`
2. APN prefix look-up for rights with no explicit geography (RBU, SFRUU, MFRUU, PRUU)

In [3]:
def apn_to_jurisdiction(apn: str) -> str:
    """Infer jurisdiction from APN prefix."""
    apn = str(apn).strip()
    if apn.startswith("1318") or apn.startswith("1418"):
        return "DG"
    try:
        prefix = int(apn.split("-")[0])
    except ValueError:
        return "TRPA"
    if 14 <= prefix <= 21 or 33 <= prefix <= 36 or prefix == 80:
        return "EL"
    if 22 <= prefix <= 32:
        return "CSLT"
    if 83 <= prefix <= 98 or 111 <= prefix <= 117:
        return "PL"
    if 122 <= prefix <= 132:
        return "WA"
    return "TRPA"


def map_jurisdiction(row) -> str:
    dr = str(row["Development Right"]).strip()
    if "El Dorado County" in dr:
        return "EL"
    if "Placer County" in dr:
        return "PL"
    if "Washoe County" in dr:
        return "WA"
    if "Douglas County" in dr:
        return "DG"
    if "CSLT" in dr or "City of South Lake Tahoe" in dr:
        return "CSLT"
    if dr == "Residential Allocation":
        return "TRPA"
    return "TRPA"


res["Jurisdiction"] = res.apply(map_jurisdiction, axis=1)
res["Jurisdiction"].value_counts()

Jurisdiction
TRPA    141
EL       39
CSLT     35
PL       18
WA       18
DG       10
Name: count, dtype: int64

## Step 3 – Map Unit_Pool

| Condition | Unit_Pool |
|---|---|
| `Development Right` is Residential Bonus Unit (RBU) | Bonus Unit |
| `Development Type` contains ADU | ADU |
| All other residential | General |

In [4]:
def map_unit_pool(row) -> str:
    dr = str(row["Development Right"]).strip()
    dt = str(row["Development Type"]).strip()
    if dr == "Residential Bonus Unit (RBU)":
        return "Bonus Unit"
    if "ADU" in dt:
        return "ADU"
    return "General"


res["Unit_Pool"] = res.apply(map_unit_pool, axis=1)
res[["APN", "Quantity", "Development Right", "Development Type", "Jurisdiction", "Unit_Pool"]].head(20)

,APN,Quantity,Development Right,Development Type,Jurisdiction,Unit_Pool
0,014-242-008,1,Residential Allocation - El Dorado County,New Single-Family Residential ADU from Allocation,EL,ADU
1,014-282-014,1,Residential Allocation - El Dorado County,New Single-Family Residential from Allocation,EL,General
2,015-032-013,1,Residential Allocation - El Dorado County,New Single-Family Residential From Allocation,EL,General
3,015-104-021,1,Residential Allocation,New Single-Family Residential From Allocation,TRPA,General
4,015-152-006,1,Residential Allocation - El Dorado County,New Single-Family Residential from Allocation,EL,General
5,015-164-012,1,Residential Allocation - El Dorado County,New Single-Family Residential from Allocation,EL,General
6,015-253-015,1,Residential Allocation - El Dorado County,New Single-Family Residential from Allocation,EL,General
7,015-265-006,1,Residential Allocation - El Dorado County,New Single-Family Residential from Allocation,EL,General
8,016-091-020,1,Residential Allocation - El Dorado County,New Single-Family Residential from Allocation,EL,General
9,016-101-067,1,Residential Allocation - El Dorado County,New Single-Family Residential from Allocation,EL,General


## Step 4 – Aggregate and write output CSV

In [7]:
res.to_csv("known_projects_assigned.csv", index=False)
summary = (
    res.groupby(["Jurisdiction", "Unit_Pool"], sort=True)["Quantity"]
    .sum()
    .reset_index()
    .rename(columns={"Quantity": "Future_Units"})
)

POOL_ORDER = {"General": 0, "Bonus Unit": 1, "ADU": 2}
summary["_pool_order"] = summary["Unit_Pool"].map(POOL_ORDER).fillna(9)
summary = (
    summary.sort_values(["Jurisdiction", "_pool_order"])
    .drop(columns="_pool_order")
    .reset_index(drop=True)
)

summary

,Jurisdiction,Unit_Pool,Future_Units
0,CSLT,General,34
1,CSLT,ADU,1
2,DG,General,9
3,DG,ADU,1
4,EL,General,38
5,EL,ADU,1
6,PL,General,20
7,PL,ADU,2
8,TRPA,General,226
9,TRPA,Bonus Unit,559


In [6]:
summary.to_csv(OUTPUT_FILE, index=False)
print(f"Saved -> {OUTPUT_FILE}")
print(f"Total Future_Units: {summary['Future_Units'].sum():,}")

Saved -> Lookup_Lists\known_projects_residential_summary.csv
Total Future_Units: 942


## QA – Compare against existing forecast_residential_zoned_units.csv

In [ ]:
reference = pd.read_csv(BASE / "forecast_residential_zoned_units.csv")

compare = (
    summary.merge(
        reference.rename(columns={"Future_Units": "Ref_Units"}),
        on=["Jurisdiction", "Unit_Pool"],
        how="outer"
    )
    .fillna(0)
    .assign(Diff=lambda d: d["Future_Units"] - d["Ref_Units"])
)

compare